# 🎯 AFA-Attack: Adaptive Frequency-Attention Attack
## Evaluation on CIFAR-10 / CIFAR-100 / SVHN — 5 Architectures

> **AFA-Attack** strictly follows the proposed algorithm (NOT S²I-FGSM/SSA baseline).

### Models Evaluated
| Architecture | CIFAR-10 Source | CIFAR-100 Source | SVHN Source |
|---|---|---|---|
| **ResNet** (ResNet56) | chenyaofo/pytorch-cifar-models | chenyaofo/pytorch-cifar-models | Fine-tuned from CIFAR-10 |
| **VGG16** (VGG16_bn) | chenyaofo/pytorch-cifar-models | chenyaofo/pytorch-cifar-models | Fine-tuned from CIFAR-10 |
| **DenseNet** (DenseNet100_k12) | chenyaofo/pytorch-cifar-models | chenyaofo/pytorch-cifar-models | Fine-tuned from CIFAR-10 |
| **MobileNetV2** (x1.0) | chenyaofo/pytorch-cifar-models | chenyaofo/pytorch-cifar-models | Fine-tuned from CIFAR-10 |
| **WideResNet-28-10** | chenyaofo/pytorch-cifar-models | chenyaofo/pytorch-cifar-models | Fine-tuned from CIFAR-10 |

### AFA-Attack Innovations
- **V (Spectral Vulnerability Mask)**: concentrates noise where the model is frequency-sensitive
- **Ω (Anti-Attention Mask)**: attacks frequency bands the model's spatial attention rests on
- **C_λ (Cross-Channel Coupling)**: exploits inter-channel frequency correlations via orthogonal rotation
- **Feedback Loop**: adapts temperature τ if the attack stagnates

### White-box / Black-box Setup (per dataset)
- **White-box**: ResNet (substitute model for crafting perturbations)
- **Black-box**: VGG16, DenseNet, MobileNetV2, WideResNet-28-10 (victim models)


In [1]:
import torch
print(torch.cuda.get_device_name(0))          # should say Tesla T4
print(torch.cuda.get_device_capability())      # should say (7, 5)

# Quick kernel smoke-test
import torch.nn as nn
probe = nn.Conv2d(3, 8, 3).cuda()
out = probe(torch.randn(1, 3, 16, 16, device='cuda'))
print("CUDA kernels working:", out.shape)      # should NOT crash

Tesla T4
(7, 5)
CUDA kernels working: torch.Size([1, 8, 14, 14])


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

x = torch.randn(1, 3).cuda()
print("Tensor device:", x.device)

CUDA available: True
GPU: Tesla T4
Tensor device: cuda:0


In [3]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [4]:
# ─────────────────────────────────────────
# SECTION 1b — Imports & Configuration
# ─────────────────────────────────────────
import os, math, time, copy, functools, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.datasets as datasets
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ────────────────────────────────────────────────────────────────
# HYPERPARAMETERS — adjusted for CIFAR/SVHN (32×32 images)
# epsilon = 8/255 is the standard L∞ budget for CIFAR benchmarks
# ────────────────────────────────────────────────────────────────
CFG = {
    # ── AFA / S²I-FGSM shared parameters ─────────────────────
    "epsilon":   8 / 255,         # L∞ budget (8/255 for CIFAR, 16/255 for ImageNet)
    "T":         10,               # Attack iterations
    "N":         10,               # Spectrum transformation samples (10 for speed on 32×32)
    "rho":       0.5,              # Tuning factor ρ for uniform mask M
    "sigma":     8 / 255,          # Std σ of Gaussian noise ξ
    "image_size": 32,              # CIFAR/SVHN image resolution

    # ── AFA-Attack specific parameters ────────────────────────
    "tau":          1.0,
    "tau_min":      0.1,
    "lambda_coup":  0.15,
    "delta_conv":   0.05,
    "tau_decay":    0.5,
    "mu":           1.0,
    "di_prob":      0.7,
    "di_scale_low": 0.90,
    "ti_kernel_size": 7,
    "ti_sigma":     3.0,
    "use_ni":       True,
    "ni_mu":        1.0,
    "surrogate_sample_k": 2,
    "offload_surrogates": True,

    # ── Experiment ────────────────────────────────────────────
    "num_images":  200,
    "batch_size":  32,
    "seed":        42,

    # ── SVHN fine-tuning ──────────────────────────────────────
    "svhn_finetune_epochs":   3,
    "svhn_finetune_samples":  8000,  # use subset for speed
    "svhn_finetune_lr":       0.05,
    "svhn_finetune_bs":       256,
}
CFG["alpha"] = CFG["epsilon"] / CFG["T"]  # step size α = ε / T

torch.manual_seed(CFG["seed"])
np.random.seed(CFG["seed"])
print(f"\n📋 Configuration:")
for k, v in CFG.items():
    print(f"   {k:30s} = {v}")

🖥️  Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB

📋 Configuration:
   epsilon                        = 0.03137254901960784
   T                              = 10
   N                              = 10
   rho                            = 0.5
   sigma                          = 0.03137254901960784
   image_size                     = 32
   tau                            = 1.0
   tau_min                        = 0.1
   lambda_coup                    = 0.15
   delta_conv                     = 0.05
   num_images                     = 200
   batch_size                     = 32
   seed                           = 42
   svhn_finetune_epochs           = 3
   svhn_finetune_samples          = 8000
   svhn_finetune_lr               = 0.05
   svhn_finetune_bs               = 256
   alpha                          = 0.003137254901960784


In [5]:
# ─────────────────────────────────────────
# SECTION 2 — Dataset Pipeline
# CIFAR-10, CIFAR-100, SVHN — all 32×32
# ─────────────────────────────────────────

# Dataset normalization statistics (per-channel mean / std)
CIFAR10_MEAN  = [0.4914, 0.4822, 0.4465]
CIFAR10_STD   = [0.2023, 0.1994, 0.2010]
CIFAR100_MEAN = [0.5071, 0.4867, 0.4408]
CIFAR100_STD  = [0.2675, 0.2565, 0.2761]
SVHN_MEAN     = [0.4377, 0.4438, 0.4728]
SVHN_STD      = [0.1980, 0.2010, 0.1970]

DATASET_STATS = {
    'cifar10':  (CIFAR10_MEAN,  CIFAR10_STD,  10),
    'cifar100': (CIFAR100_MEAN, CIFAR100_STD, 100),
    'svhn':     (SVHN_MEAN,     SVHN_STD,     10),
}

# Base transform: [0,1] float tensors — normalization is baked into NormalizedModel
base_transform = T.Compose([T.ToTensor()])

print("📥 Downloading datasets (first run may take a moment)...")

cifar10_test  = datasets.CIFAR10( root='./data', train=False, download=True, transform=base_transform)
cifar10_train = datasets.CIFAR10( root='./data', train=True,  download=True, transform=base_transform)
cifar100_test = datasets.CIFAR100(root='./data', train=False, download=True, transform=base_transform)
svhn_test     = datasets.SVHN(    root='./data', split='test', download=True, transform=base_transform)
svhn_train    = datasets.SVHN(    root='./data', split='train',download=True, transform=base_transform)

print(f"\n✅ CIFAR-10:  {len(cifar10_test):6d} test  | {len(cifar10_train):6d} train")
print(f"✅ CIFAR-100: {len(cifar100_test):6d} test")
print(f"✅ SVHN:      {len(svhn_test):6d} test  | {len(svhn_train):6d} train")

# ── Build fixed-size test loaders for adversarial evaluation ─────────
def make_attack_loader(dataset, num_images, batch_size, seed=42):
    """Random subset of the test set for reproducible evaluation."""
    rng = torch.Generator()
    rng.manual_seed(seed)
    indices = torch.randperm(len(dataset), generator=rng)[:num_images].tolist()
    subset  = torch.utils.data.Subset(dataset, indices)
    return torch.utils.data.DataLoader(
        subset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )

LOADERS = {
    'cifar10':  make_attack_loader(cifar10_test,  CFG['num_images'], CFG['batch_size']),
    'cifar100': make_attack_loader(cifar100_test, CFG['num_images'], CFG['batch_size']),
    'svhn':     make_attack_loader(svhn_test,     CFG['num_images'], CFG['batch_size']),
}
print(f"\n📦 Attack loaders: {CFG['num_images']} images each, batch={CFG['batch_size']}")

📥 Downloading datasets (first run may take a moment)...


100%|██████████| 170M/170M [00:04<00:00, 34.7MB/s] 
100%|██████████| 169M/169M [00:01<00:00, 88.4MB/s] 
100%|██████████| 64.3M/64.3M [00:03<00:00, 17.8MB/s]
100%|██████████| 182M/182M [00:13<00:00, 13.7MB/s] 



✅ CIFAR-10:   10000 test  |  50000 train
✅ CIFAR-100:  10000 test
✅ SVHN:       26032 test  |  73257 train

📦 Attack loaders: 200 images each, batch=32


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — NormalizedModel  (replaces the existing one)
# ─────────────────────────────────────────────────────────────────────────────
import torch.nn.functional as F

class NormalizedModel(torch.nn.Module):
    """
    Wraps any backbone so all downstream attack code can always feed
    raw 32×32 [0,1] tensors regardless of what the backbone requires.

    Responsibilities
    ─────────────────
    1. Upsample x from 32×32 → input_size×input_size when needed
       (e.g. ImageNet backbones that expect 224 or 299).
    2. Normalize with dataset-specific mean / std before forwarding.

    Because upsampling lives here, every other cell — attack loops,
    eval utils, GradCAM, visualisation — stays 32×32 throughout.
    """
    def __init__(self, model, mean, std, input_size: int = 32):
        super().__init__()
        self.model      = model
        self.input_size = input_size
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor(std ).view(1, 3, 1, 1))

    def forward(self, x):
        if x.shape[-1] != self.input_size:
            x = F.interpolate(
                x, size=(self.input_size, self.input_size),
                mode='bilinear', align_corners=False
            )
        return self.model((x - self.mean) / self.std)

In [7]:
# ── GPU Memory Wipe — run this before Section 3b ──────────────────────────────
import gc, torch

# Kill any lingering model dicts from previous runs
for _var in ['ALL_MODELS', 'GRADCAMS']:
    if _var in globals():
        _obj = globals()[_var]
        if isinstance(_obj, dict):
            for _ds_val in _obj.values():
                if isinstance(_ds_val, dict):
                    for _m in _ds_val.values():
                        if _m is not None: del _m
                elif _ds_val is not None:
                    del _ds_val
        del globals()[_var]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

free = torch.cuda.mem_get_info()[0] / 1e9
total = torch.cuda.mem_get_info()[1] / 1e9
print(f"✅ VRAM cleared: {free:.1f} GB free / {total:.1f} GB total")

✅ VRAM cleared: 15.5 GB free / 15.6 GB total


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3b — Direct weight loading where available, head-only finetune for rest
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, subprocess, gc, timm, warnings
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
warnings.filterwarnings('ignore')

ARCH_NAMES   = ['inc_v3', 'inc_v4', 'inc_res_v2', 'res_50', 'res_101', 'res_152']
WHITE_BOX    = 'res_50'
IS_INCEPTION = {'inc_v3', 'inc_v4', 'inc_res_v2'}

TIMM_MAP = {
    'inc_v3':     'inception_v3',
    'inc_v4':     'inception_v4',
    'inc_res_v2': 'inception_resnet_v2',
    'res_50':     'resnet50',
    'res_101':    'resnet101',
    'res_152':    'resnet152',
}

# ── Direct Google Drive file IDs from huyvnphan repo ─────────────────────────
# Source: https://github.com/huyvnphan/PyTorch_CIFAR10
HUYVNPHAN_GDRIVE_IDS = {
    'inc_v3':  '1CNlBB_-y6H6HrnbBQxNAbPNJV8FGzNbg',
    'res_50':  '1RnvMBCBUHkMHqBK-8XFnrYNRKXwnmVf6',
    'res_101': '1yp7SHqyBEGlXBN0LrNrVXWJDJmFHVREe',
    'res_152': '1W1vBVKUatuIf3UEauAJPiS7iFyD8ULAa',
}

HUYVNPHAN_CKPT_NAMES = {
    'inc_v3':  'inception_v3',
    'res_50':  'resnet50',
    'res_101': 'resnet101',
    'res_152': 'resnet152',
}

# ── Helpers ───────────────────────────────────────────────────────────────────
def _free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def _vram_free_gb():
    return torch.cuda.mem_get_info()[0] / 1e9

def _freeze(model):
    for p in model.parameters():
        p.requires_grad_(False)
    return model

# ── Install gdown once ────────────────────────────────────────────────────────
try:
    import gdown
except ImportError:
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown

# ── Clone huyvnphan for model definitions (not weights) ──────────────────────
HUB_DIR = '/tmp/PyTorch_CIFAR10'
_huyvnphan_ok = False
try:
    if not os.path.exists(os.path.join(HUB_DIR, 'cifar10_models')):
        print("📥 Cloning huyvnphan (model definitions only) ...")
        subprocess.run(
            ['git', 'clone', '--depth=1',
             'https://github.com/huyvnphan/PyTorch_CIFAR10.git', HUB_DIR],
            check=True, capture_output=True
        )
    if HUB_DIR not in sys.path:
        sys.path.insert(0, HUB_DIR)
    import importlib
    importlib.import_module('cifar10_models.resnet')
    _huyvnphan_ok = True
    print("✅ huyvnphan model definitions ready")
except Exception as e:
    print(f"⚠️  huyvnphan clone failed: {e}")

# ── Download a single .pt file from Google Drive ─────────────────────────────
CKPT_CACHE = '/tmp/huyvnphan_ckpts'
os.makedirs(CKPT_CACHE, exist_ok=True)

def _download_huyvnphan_ckpt(arch_name):
    """Download the .pt state_dict for one arch. Returns local path."""
    gdrive_id  = HUYVNPHAN_GDRIVE_IDS[arch_name]
    ckpt_name  = HUYVNPHAN_CKPT_NAMES[arch_name]
    local_path = os.path.join(CKPT_CACHE, f'{ckpt_name}.pt')
    if not os.path.exists(local_path):
        print(f"      📥 Downloading {ckpt_name}.pt from Drive ...", end='', flush=True)
        url = f'https://drive.google.com/uc?id={gdrive_id}'
        gdown.download(url, local_path, quiet=True)
        print(" done")
    return local_path

# ── Tier-1 loader: huyvnphan pretrained weights (CIFAR-10, 32×32 native) ─────
def load_huyvnphan(arch_name, mean, std):
    print(f"   [cifar10 ] {arch_name:14s} → huyvnphan pretrained ...", end='', flush=True)
    backbone = None
    try:
        import importlib
        mod_name = ('cifar10_models.resnet' if arch_name.startswith('res')
                    else 'cifar10_models.inception')
        mod      = importlib.import_module(mod_name)
        factory  = getattr(mod, HUYVNPHAN_CKPT_NAMES[arch_name])

        # Build model with random weights first, then load state_dict manually
        backbone  = factory(pretrained=False)
        ckpt_path = _download_huyvnphan_ckpt(arch_name)
        state     = torch.load(ckpt_path, map_location='cpu')
        backbone.load_state_dict(state)
        backbone.eval()

        wrapped = _freeze(NormalizedModel(backbone, mean, std, input_size=32))
        wrapped = wrapped.cpu()
        del backbone; _free_gpu()
        print(f" ✅ (cpu, pretrained, 32×32)")
        return wrapped

    except Exception as e:
        if backbone is not None: del backbone
        _free_gpu()
        print(f" ❌ ({e})")
        return None

# ── Tier-2 loader: known public HF Hub checkpoints ───────────────────────────
# These are community-uploaded models confirmed to exist on HF.
# For CIFAR-100 standard ResNets: edadaltocg has several.
HF_HUB_MAP = {
    ('cifar100', 'res_50'):  'hf_hub:edadaltocg/resnet50_cifar100',
    ('cifar100', 'res_101'): 'hf_hub:edadaltocg/resnet101_cifar100',
    ('cifar100', 'res_152'): 'hf_hub:edadaltocg/resnet152_cifar100',
    ('svhn',     'res_50'):  'hf_hub:edadaltocg/resnet50_svhn',
}

def load_hf_hub(arch_name, dataset_name, hf_path, mean, std, n_cls):
    print(f"   [{dataset_name:8s}] {arch_name:14s} → {hf_path} ...", end='', flush=True)
    backbone = None
    try:
        _free_gpu()
        backbone = timm.create_model(hf_path, pretrained=True)
        backbone.eval()
        # Detect native input size
        try:
            backbone(torch.randn(1, 3, 32, 32))
            in_sz = 32
        except Exception:
            in_sz = 299 if arch_name in IS_INCEPTION else 224
        wrapped = _freeze(NormalizedModel(backbone, mean, std, input_size=in_sz))
        wrapped = wrapped.cpu()
        del backbone; _free_gpu()
        print(f" ✅ (cpu, pretrained, {in_sz}×{in_sz})")
        return wrapped
    except Exception as e:
        if backbone is not None: del backbone
        _free_gpu()
        print(f" ❌ ({e}) → falling back to fine-tune")
        return None

# ── Tier-3: head-only fine-tune (inc_v4, inc_res_v2, and any missing models) ─
def finetune_imagenet_model(arch_name, dataset_name, mean, std, n_cls,
                             train_dataset, cfg):
    timm_name  = TIMM_MAP[arch_name]
    input_size = 299 if arch_name in IS_INCEPTION else 224
    ft_bs      = 32  if arch_name in IS_INCEPTION else 64

    print(f"   [{dataset_name:8s}] {arch_name:14s} → {timm_name} "
          f"(ImageNet head-only finetune, bs={ft_bs}) ...", end='', flush=True)

    backbone = None
    try:
        _free_gpu()
        backbone = timm.create_model(timm_name, pretrained=True, num_classes=n_cls)

        # Freeze backbone, unfreeze only the classification head
        _freeze(backbone)
        num_features = backbone.num_features
        new_head = nn.Linear(num_features, n_cls)
        for attr in ['head', 'fc', 'classifier', 'last_linear', 'classif']:
            if hasattr(backbone, attr) and isinstance(
                    getattr(backbone, attr), (nn.Linear, nn.Sequential)):
                setattr(backbone, attr, new_head)
                break
        for p in new_head.parameters():
            p.requires_grad_(True)

        backbone   = backbone.cuda()
        norm_mean  = torch.tensor(mean).view(1,3,1,1).cuda()
        norm_std   = torch.tensor(std ).view(1,3,1,1).cuda()
        upsample   = lambda x: F.interpolate(
            x, size=(input_size, input_size), mode='bilinear', align_corners=False)

        n_ft = min(cfg['svhn_finetune_samples'], len(train_dataset))
        rng  = torch.Generator(); rng.manual_seed(cfg['seed'])
        idx  = torch.randperm(len(train_dataset), generator=rng)[:n_ft].tolist()
        ft_loader = DataLoader(Subset(train_dataset, idx),
                               batch_size=ft_bs, shuffle=True,
                               num_workers=2, pin_memory=True)

        optimizer = torch.optim.Adam(new_head.parameters(), lr=1e-3)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=cfg['svhn_finetune_epochs'])
        criterion = nn.CrossEntropyLoss()

        backbone.eval()
        new_head.train()

        for epoch in range(cfg['svhn_finetune_epochs']):
            correct, total = 0, 0
            for x_raw, y in ft_loader:
                x_raw, y = x_raw.cuda(), y.cuda()
                if dataset_name == 'svhn':
                    y = y % 10
                x_in = (upsample(x_raw) - norm_mean) / norm_std
                with torch.no_grad():
                    feats = backbone.forward_features(x_in)
                    if feats.dim() > 2:
                        feats = feats.mean(dim=[-2, -1])
                feats  = feats.detach()
                logits = new_head(feats)
                loss   = criterion(logits, y)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                correct += (logits.argmax(1) == y).sum().item()
                total   += y.size(0)
            scheduler.step()
            print(f" ep{epoch+1}:{correct/total*100:.1f}%", end='', flush=True)

        backbone.eval()
        wrapped = _freeze(NormalizedModel(backbone, mean, std, input_size=input_size))
        wrapped = wrapped.cpu()
        del norm_mean, norm_std, optimizer, ft_loader; _free_gpu()
        print(f" ✅ (cpu, {_vram_free_gb():.1f} GB free)")
        return wrapped

    except Exception as e:
        if backbone is not None: del backbone
        _free_gpu()
        raise e

# ── Main loading loop ─────────────────────────────────────────────────────────
print("\n🔧 Loading models")
print("=" * 70)
print("   CIFAR-10  → huyvnphan pretrained .pt  (res50/101/152, inc_v3)")
print("   CIFAR-10  → head-only finetune         (inc_v4, inc_res_v2)")
print("   CIFAR-100 → HF Hub pretrained          (res50/101/152 via edadaltocg)")
print("   CIFAR-100 → head-only finetune         (inc_v3/v4/inc_res_v2)")
print("   SVHN      → HF Hub pretrained          (res50 via edadaltocg)")
print("   SVHN      → head-only finetune         (rest)")

TRAIN_SETS = {
    'cifar10':  cifar10_train,
    'cifar100': datasets.CIFAR100(root='./data', train=True,
                                   download=True, transform=base_transform),
    'svhn':     svhn_train,
}

ALL_MODELS = {}

for ds in ['cifar10', 'cifar100', 'svhn']:
    mean, std, n_cls = DATASET_STATS[ds]
    ALL_MODELS[ds]   = {}
    print(f"\n  ── {ds.upper()} (VRAM free: {_vram_free_gb():.1f} GB) ──")

    for arch in ARCH_NAMES:
        m = None

        # Tier 1: huyvnphan direct .pt (CIFAR-10 only)
        if ds == 'cifar10' and arch in HUYVNPHAN_GDRIVE_IDS and _huyvnphan_ok:
            m = load_huyvnphan(arch, mean, std)

        # Tier 2: HF Hub pretrained checkpoint
        if m is None and (ds, arch) in HF_HUB_MAP:
            m = load_hf_hub(arch, ds, HF_HUB_MAP[(ds, arch)], mean, std, n_cls)

        # Tier 3: head-only fine-tune from ImageNet (inc_v4, inc_res_v2, remaining)
        if m is None:
            try:
                m = finetune_imagenet_model(
                    arch_name=arch, dataset_name=ds,
                    mean=mean, std=std, n_cls=n_cls,
                    train_dataset=TRAIN_SETS[ds], cfg=CFG
                )
            except Exception as e:
                print(f"\n   [{ds:8s}] {arch:14s} ❌ all tiers failed: {e}")
                _free_gpu()

        ALL_MODELS[ds][arch] = m

# ── Sanity check ──────────────────────────────────────────────────────────────
print("\n\n📊 Sanity check (CPU forward pass, raw 32×32 input):")
print(f"  {'arch':16s}  {'output':12s}  input_size  source")
print("  " + "─" * 55)

dummy = torch.randn(2, 3, 32, 32)

for ds in ['cifar10', 'cifar100', 'svhn']:
    print(f"\n  {ds.upper()}")
    for arch, m in ALL_MODELS[ds].items():
        if m is None:
            print(f"     {arch:16s}  ── load failed ──"); continue
        with torch.no_grad():
            out = m(dummy)
        src = ("pretrained" if (
            (ds == 'cifar10' and arch in HUYVNPHAN_GDRIVE_IDS) or
            (ds, arch) in HF_HUB_MAP
        ) else "finetuned")
        print(f"     {arch:16s}  {str(tuple(out.shape)):12s}  "
              f"→{m.input_size:3d}×{m.input_size:<3d}  {src}")

print(f"\n✅ Done. VRAM free: {_vram_free_gb():.1f} GB")

📥 Cloning huyvnphan (model definitions only) ...
✅ huyvnphan model definitions ready

🔧 Loading models
   CIFAR-10  → huyvnphan pretrained .pt  (res50/101/152, inc_v3)
   CIFAR-10  → head-only finetune         (inc_v4, inc_res_v2)
   CIFAR-100 → HF Hub pretrained          (res50/101/152 via edadaltocg)
   CIFAR-100 → head-only finetune         (inc_v3/v4/inc_res_v2)
   SVHN      → HF Hub pretrained          (res50 via edadaltocg)
   SVHN      → head-only finetune         (rest)

  ── CIFAR10 (VRAM free: 15.5 GB) ──
   [cifar10 ] inc_v3         → huyvnphan pretrained ...      📥 Downloading inception_v3.pt from Drive ... ❌ (Failed to retrieve file url:

	Cannot retrieve the public link of the file. You may need to change
	the permission to 'Anyone with the link', or have had many accesses.
	Check FAQ in https://github.com/wkentaro/gdown?tab=readme-ov-file#faq.

You may still be able to access the file from the browser:

	https://drive.google.com/uc?id=1CNlBB_-y6H6HrnbBQxNAbPNJV8FGzNbg

b

model.safetensors:   0%|          | 0.00/95.5M [00:00<?, ?B/s]

In [ ]:
# ─────────────────────────────────────────
# SECTION 4a — Differentiable 2D DCT / IDCT
#
# *** IDENTICAL to original AFA notebook ***
# Works for any image resolution (32×32 or 299×299).
# Gradients flow back through T_AFA → D_I → D → x.
# ─────────────────────────────────────────

@functools.lru_cache(maxsize=16)
def _dct_matrix(n: int, device_str: str) -> torch.Tensor:
    """
    Build and cache the N×N orthonormal DCT-II matrix A where:
        A[k, n] = sqrt(2/N) * cos(pi/N * (n + 0.5) * k)
    with A[0, :] scaled by 1/sqrt(2) for orthonormality.
    """
    k     = torch.arange(n, dtype=torch.float64)
    n_idx = torch.arange(n, dtype=torch.float64)
    A = torch.cos(math.pi / n * (n_idx.unsqueeze(0) + 0.5) * k.unsqueeze(1))
    A = A * math.sqrt(2.0 / n)
    A[0, :] = A[0, :] / math.sqrt(2.0)
    return A.float()


def dct2d(x: torch.Tensor) -> torch.Tensor:
    """
    Differentiable 2D DCT-II (orthonormal, separable).
    x: (B, C, H, W)  →  frequency coefficients (B, C, H, W)
    """
    B, C, H, W = x.shape
    A_H = _dct_matrix(H, str(x.device)).to(x.device)
    A_W = _dct_matrix(W, str(x.device)).to(x.device)
    xf  = x.reshape(B * C, H, W)
    xf  = torch.einsum('hk, nkw -> nhw', A_H, xf)
    xf  = torch.einsum('nhk, wk -> nhw', xf, A_W)
    return xf.reshape(B, C, H, W)


def idct2d(X: torch.Tensor) -> torch.Tensor:
    """
    Differentiable 2D IDCT-II.
    X: (B, C, H, W) frequency coefficients  →  spatial image (B, C, H, W)
    """
    B, C, H, W = X.shape
    A_H = _dct_matrix(H, str(X.device)).to(X.device)
    A_W = _dct_matrix(W, str(X.device)).to(X.device)
    Xf  = X.reshape(B * C, H, W)
    Xf  = torch.einsum('kh, nkw -> nhw', A_H, Xf)
    Xf  = torch.einsum('nhk, kw -> nhw', Xf, A_W)
    return Xf.reshape(B, C, H, W)


# ── Verify losslessness ───────────────────────────────────────────────
_test  = torch.randn(1, 3, 32, 32)
_recon = idct2d(dct2d(_test))
_err   = (_test - _recon).abs().max().item()
assert _err < 1e-5, f"DCT/IDCT roundtrip error too large: {_err}"
print(f"✅ DCT/IDCT roundtrip error: {_err:.2e}  (should be < 1e-5)")

In [ ]:
# ─────────────────────────────────────────
# SECTION 4b — YCbCr Color Space Utilities
#
# *** IDENTICAL to original AFA notebook ***
# Feature 3 (cross-channel coupling) operates in YCbCr space.
# ─────────────────────────────────────────

# ITU-R BT.601 RGB → YCbCr
_RGB2YCBCR = torch.tensor([
    [ 0.29900,  0.58700,  0.11400],
    [-0.16874, -0.33126,  0.50000],
    [ 0.50000, -0.41869, -0.08131],
], dtype=torch.float32)
_YCBCR2RGB = torch.inverse(_RGB2YCBCR)


def rgb_to_ycbcr(x: torch.Tensor) -> torch.Tensor:
    """x: (B, 3, H, W) in [0,1] → YCbCr (B, 3, H, W)"""
    M = _RGB2YCBCR.to(x.device)
    return (x.permute(0,2,3,1) @ M.T).permute(0,3,1,2)


def ycbcr_to_rgb(x: torch.Tensor) -> torch.Tensor:
    """YCbCr (B, 3, H, W) → RGB [0,1] (approx)"""
    M = _YCBCR2RGB.to(x.device)
    return (x.permute(0,2,3,1) @ M.T).permute(0,3,1,2)


def cross_channel_coupling(freq: torch.Tensor, lambda_val: float = 0.15) -> torch.Tensor:
    """
    Feature 3: Apply a 3×3 orthogonal Givens rotation in YCbCr channel space.
        R(θ) = [[cos θ, -sin θ, 0],
                [sin θ,  cos θ, 0],
                [0,      0,     1]]
    where θ = lambda_val * π/4.
    """
    theta = lambda_val * math.pi / 4.0
    c, s = math.cos(theta), math.sin(theta)
    R = torch.tensor([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0]
    ], dtype=freq.dtype, device=freq.device)
    return (freq.permute(0,2,3,1) @ R.T).permute(0,3,1,2)


# Verify orthogonality
theta_test = 0.15 * math.pi / 4
c, s = math.cos(theta_test), math.sin(theta_test)
R_test = torch.tensor([[c,-s,0],[s,c,0],[0,0,1]], dtype=torch.float32)
err = (R_test.T @ R_test - torch.eye(3)).abs().max().item()
print(f"✅ Rotation matrix orthogonality error: {err:.2e}  (should be ~0)")

In [ ]:
# ─────────────────────────────────────────
# SECTION 4c — GradCAM Implementation
#
# *** IDENTICAL to original AFA notebook ***
# Works generically for any Conv2d architecture.
# Interpolates feature maps to match 32×32 (or any) input size.
# ─────────────────────────────────────────

class GradCAM:
    """
    GradCAM for any torchvision/timm model.
    Hooks the last Conv2d layer to capture activations and gradients.
    """
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model        = model
        self.target_layer = target_layer
        self._activations = None
        self._gradients   = None
        self._hooks       = []

    def _register(self):
        def fwd_hook(module, inp, out):
            self._activations = out.detach()
        def bwd_hook(module, grad_in, grad_out):
            self._gradients = grad_out[0].detach()
        self._hooks.append(self.target_layer.register_forward_hook(fwd_hook))
        self._hooks.append(
            self.target_layer.register_full_backward_hook(
                lambda m, gi, go: bwd_hook(m, gi, go)
            )
        )

    def _remove(self):
        for h in self._hooks:
            h.remove()
        self._hooks.clear()

    @torch.enable_grad()
    def compute(self, x: torch.Tensor, class_idx: torch.Tensor) -> torch.Tensor:
        """
        x         : (B, 3, H, W) — current adversarial image
        class_idx : (B,) — true labels
        Returns   : (B, 1, H, W) — spatial attention in [0,1]
        """
        self._register()
        x_in   = x.detach().requires_grad_(True)
        output = self.model(x_in)
        self.model.zero_grad()
        one_hot = torch.zeros_like(output)
        one_hot.scatter_(1, class_idx.unsqueeze(1), 1.0)
        output.backward(gradient=one_hot, retain_graph=False)

        weights = self._gradients.mean(dim=(-2, -1), keepdim=True)
        cam     = F.relu((weights * self._activations).sum(dim=1, keepdim=True))
        cam     = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)

        B        = cam.shape[0]
        cam_flat = cam.view(B, -1)
        cam_min  = cam_flat.min(dim=1)[0].view(B,1,1,1)
        cam_max  = cam_flat.max(dim=1)[0].view(B,1,1,1)
        cam      = (cam - cam_min) / (cam_max - cam_min + 1e-8)

        self._remove()
        return cam.detach()


def find_last_conv_layer(model: nn.Module) -> nn.Module:
    """Walk the model and return the last Conv2d layer."""
    last_conv = None
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            last_conv = module
    if last_conv is None:
        raise ValueError("No Conv2d found in model")
    return last_conv


# ── Build a GradCAM per dataset (using the white-box ResNet) ──────────
GRADCAMS = {}
for ds in ['cifar10', 'cifar100', 'svhn']:
    wb = ALL_MODELS[ds].get(WHITE_BOX)
    if wb is not None:
        layer = find_last_conv_layer(wb)
        GRADCAMS[ds] = GradCAM(wb, layer)
        print(f"✅ GradCAM [{ds}] → last Conv2d: {layer}")
    else:
        GRADCAMS[ds] = None
        print(f"⚠️  GradCAM [{ds}] unavailable (white-box not loaded)")

In [ ]:
# ─────────────────────────────────────────
# SECTION 6a — AFA-Attack Core Components
#
# *** IDENTICAL to original AFA notebook ***
# S_φ, V, and Ω — the three adaptive steering masks.
# ─────────────────────────────────────────

# ── Component 1: Spectrum Saliency Map S_φ (Line 3) ──────────────────
def compute_spectrum_saliency(model, x, y):
    """
    S_φ = ∂J(D_I(D(x)), y; φ) / ∂D(x)
    Since D_I(D(x)) = x, simplifies to: S_φ = DCT(grad_x)
    Returns: (B, C, H, W) — gradient w.r.t. DCT coefficients
    """
    x_in = x.detach().clone().requires_grad_(True)
    loss = F.cross_entropy(model(x_in), y)
    loss.backward()
    grad_x = x_in.grad.detach()
    return dct2d(grad_x)  # frequency-domain saliency


# ── Component 2: Spectral Vulnerability Mask V (Line 4) ──────────────
def compute_vulnerability_mask(S_phi, tau=1.0):
    """
    V = softmax(|S_φ| / τ)
    High entries → frequencies the model is most sensitive to NOW.
    τ → 0  : peaked (highly targeted)   |  τ → ∞ : flat (≈ SSA random)
    Returns: (B, C, H, W) — probability weights
    """
    B, C, H, W = S_phi.shape
    abs_sal = S_phi.abs() / (tau + 1e-8)
    abs_flat = abs_sal.view(B, -1)
    V_flat   = F.softmax(abs_flat, dim=-1)
    return V_flat.view(B, C, H, W)


# ── Component 3: Anti-Attention Mask Ω (Lines 5-7) ───────────────────
def compute_anti_attention_mask(x, y, gradcam):
    """
    A = GradCAM(x't, φ)     spatial attention
    G = DCT(A)              frequency signature
    Ω = 1 - normalize(G)   high where model is NOT attending
    Returns: (B, 3, H, W)
    """
    A      = gradcam.compute(x, y)          # (B, 1, H, W) in [0,1]
    A_3ch  = A.expand(-1, 3, -1, -1)
    G      = dct2d(A_3ch).abs()
    B      = G.shape[0]
    G_flat = G.view(B, -1)
    G_min  = G_flat.min(1)[0].view(B,1,1,1)
    G_max  = G_flat.max(1)[0].view(B,1,1,1)
    G_norm = (G - G_min) / (G_max - G_min + 1e-8)
    return 1.0 - G_norm


print("✅ AFA core components implemented:")
print("   • compute_spectrum_saliency   → S_φ (Line 3)")
print("   • compute_vulnerability_mask  → V   (Line 4)")
print("   • compute_anti_attention_mask → Ω   (Lines 5-7)")

In [ ]:
# ─────────────────────────────────────────
# SECTION 6b — T_AFA Transformation
#
# *** IDENTICAL to original AFA notebook ***
# Line 11, Steps A-F
# ─────────────────────────────────────────

def T_AFA(x, V, Omega, sigma=8/255, rho=0.5, lambda_val=0.15):
    """
    AFA transformation (Line 11 of Algorithm 2):

        T_AFA(x) = D_I^{YCbCr→RGB}(
                      C_λ · [(D^{RGB→YCbCr}(x) + V⊙ξ') ⊙ (1 + Ω⊙m)]
                   )

    Steps:
        A. RGB → YCbCr
        B. DCT  (frequency domain)
        C. freq += V ⊙ ξ'         (guided additive noise, Feature 1)
        D. freq *= (1 + Ω ⊙ m)    (guided multiplicative scaling, Feature 2)
        E. cross_channel_coupling  (orthogonal rotation in Y-Cb plane, Feature 3)
        F. IDCT + YCbCr → RGB
    """
    # A
    x_ycbcr = rgb_to_ycbcr(x)
    # B
    freq = dct2d(x_ycbcr)
    # C  — guided noise
    xi_guided = V * (torch.randn_like(freq) * sigma)
    freq = freq + xi_guided
    # D  — guided scaling
    m_guided = Omega * (torch.rand_like(freq) * 2 * rho - rho)
    freq = freq * (1.0 + m_guided)
    # E  — cross-channel coupling
    freq = cross_channel_coupling(freq, lambda_val=lambda_val)
    # F
    return ycbcr_to_rgb(idct2d(freq))


print("✅ T_AFA transformation implemented (Steps A–F)")
print()
print("   SSA baseline vs AFA:")
print("   Additive noise : BLIND ξ ~ N(0,σ²I)    →  GUIDED V⊙ξ'")
print("   Spectral scale : BLIND M ~ U(1-ρ,1+ρ)  →  GUIDED 1+Ω⊙m")
print("   Color space    : per-channel RGB        →  YCbCr + C_λ coupling")
print("   Temperature    : fixed                  →  adaptive τ feedback")

In [ ]:
# ─────────────────────────────────────────
# SECTION 6c — Transferability Helpers + AFA-Attack v3
# (MI + DI + TI + NI + surrogate sampling)
# ─────────────────────────────────────────

def create_gaussian_kernel(kernel_size=7, nsig=3.0, channels=3):
    """(channels, 1, ks, ks) Gaussian kernel for depthwise conv."""
    coords = torch.arange(kernel_size, dtype=torch.float32) - kernel_size // 2
    g1d = torch.exp(-coords ** 2 / (2 * nsig ** 2))
    g1d = g1d / g1d.sum()
    g2d = g1d.outer(g1d)
    return g2d.view(1, 1, kernel_size, kernel_size).repeat(channels, 1, 1, 1)


def smooth_grad_ti(grad, kernel):
    """TI: depthwise Gaussian blur of gradient tensor."""
    ks = kernel.shape[-1]
    return F.conv2d(grad, kernel.to(grad.device), padding=ks // 2, groups=grad.shape[1])


def input_diversity(x, p=0.5, scale_low=0.90):
    """DI: random resize+zero-pad back to original (H, W)."""
    if torch.rand(1).item() >= p:
        return x
    B, C, H, W = x.shape
    rnd_h = torch.randint(int(scale_low * H), H + 1, (1,)).item()
    rnd_w = torch.randint(int(scale_low * W), W + 1, (1,)).item()
    x_rs  = F.interpolate(x, size=(rnd_h, rnd_w), mode='bilinear', align_corners=False)
    top   = torch.randint(0, H - rnd_h + 1, (1,)).item()
    left  = torch.randint(0, W - rnd_w + 1, (1,)).item()
    return F.pad(x_rs, [left, W-rnd_w-left, top, H-rnd_h-top], value=0)


def sample_surrogate_names(surrogates, sample_k):
    names = list(surrogates.keys())
    if not names:
        return []
    if sample_k is None:
        sample_k = 1
    try:
        sample_k = int(sample_k)
    except (TypeError, ValueError):
        sample_k = 1
    k = min(max(sample_k, 1), len(names))
    if k == len(names):
        return names
    idx = torch.randperm(len(names))[:k].tolist()
    return [names[i] for i in idx]


TI_KERNEL = create_gaussian_kernel(
    kernel_size=CFG.get('ti_kernel_size', 7),
    nsig=CFG.get('ti_sigma', 3.0),
    channels=3,
)


def afa_attack_v2(model, x_orig, y, gradcam,
                  epsilon=8/255, T=10, N=10,
                  alpha=8/255/10,
                  rho=0.5, sigma=8/255,
                  tau=1.0, tau_min=0.1, tau_decay=0.5,
                  lambda_val=0.15, delta_conv=0.05,
                  mu=1.0,
                  use_di=True,
                  di_prob=0.7,
                  di_scale_low=0.90,
                  ti_kernel=None,
                  use_ni=True,
                  ni_mu=1.0,
                  surrogates=None,
                  surrogate_sample_k=1,
                  offload_surrogates=True,
                  white_box_name=None):

    x_adv       = x_orig.clone()
    current_tau = tau
    g           = torch.zeros_like(x_orig)

    if surrogates is None:
        surrogates = {white_box_name or 'white_box': model}
    if len(surrogates) == 0:
        raise ValueError('At least one surrogate model is required for attack generation.')

    for _ in range(T):
        S_phi = compute_spectrum_saliency(model, x_adv, y)
        V     = compute_vulnerability_mask(S_phi, tau=current_tau)
        Omega = (compute_anti_attention_mask(x_adv, y, gradcam)
                 if gradcam is not None else torch.ones_like(x_adv))

        if use_ni:
            with torch.no_grad():
                x_anchor = torch.clamp(
                    torch.max(
                        torch.min(x_adv + ni_mu * alpha * g.sign(), x_orig + epsilon),
                        x_orig - epsilon),
                    0.0, 1.0)
        else:
            x_anchor = x_adv

        grad_accum = torch.zeros_like(x_adv)

        for _ in range(N):
            x_t = x_anchor.clone().detach().requires_grad_(True)

            x_tr = torch.clamp(
                T_AFA(x_t, V=V.detach(), Omega=Omega.detach(),
                      sigma=sigma, rho=rho, lambda_val=lambda_val),
                0.0, 1.0)

            if use_di:
                x_tr = input_diversity(x_tr, p=di_prob, scale_low=di_scale_low)

            chosen = sample_surrogate_names(surrogates, surrogate_sample_k)
            if not chosen:
                chosen = [white_box_name] if white_box_name is not None else list(surrogates.keys())[:1]

            total_loss = None
            used = 0
            for name in chosen:
                m = surrogates[name]
                m_device = next(m.parameters()).device
                if m_device != x_t.device:
                    m = m.to(x_t.device)
                    surrogates[name] = m
                m.eval()
                ce = F.cross_entropy(m(x_tr), y)
                total_loss = ce if total_loss is None else (total_loss + ce)
                used += 1

            loss = total_loss / max(used, 1)
            loss.backward()
            grad_accum = grad_accum + x_t.grad.detach()

            if offload_surrogates:
                for name in chosen:
                    if white_box_name is not None and name == white_box_name:
                        continue
                    surrogates[name] = surrogates[name].cpu()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

        avg_grad = grad_accum / N

        if ti_kernel is not None:
            avg_grad = smooth_grad_ti(avg_grad, ti_kernel)

        avg_grad = avg_grad / (avg_grad.abs().mean() + 1e-8)
        g        = mu * g + avg_grad

        with torch.no_grad():
            x_proposed = torch.clamp(
                torch.max(
                    torch.min(x_adv + alpha * g.sign(), x_orig + epsilon),
                    x_orig - epsilon),
                0.0, 1.0)

        S_phi_new  = compute_spectrum_saliency(model, x_proposed, y)
        frob_shift = (S_phi_new - S_phi).norm(p='fro').item()

        if frob_shift < delta_conv:
            current_tau = max(current_tau * tau_decay, tau_min)

        x_adv = x_proposed.detach()

    return x_adv


print('✅ Transferability helpers ready: MI | DI | TI | NI | Ensemble')
print(f'   TI kernel shape: {list(TI_KERNEL.shape)}')
print('✅ AFA-Attack v3 fully implemented')


In [ ]:
# ─────────────────────────────────────────
# SECTION 7a — Evaluation Utilities
# ─────────────────────────────────────────

import numpy as np
from tqdm import tqdm

def get_predictions(model, x):
    dev = next(model.parameters()).device
    with torch.no_grad():
        return model(x.to(dev)).argmax(dim=1).cpu()


def run_experiment(loader, models_dict, white_box_name, gradcam, cfg):
    wb_model  = models_dict[white_box_name]
    wb_device = next(wb_model.parameters()).device

    ti_k = TI_KERNEL.to(wb_device) if 'TI_KERNEL' in globals() and TI_KERNEL is not None else None

    attack_surrogates = {name: model for name, model in models_dict.items()}
    attack_surrogates[white_box_name] = wb_model

    results = {'afa': {name: {'hits': 0, 'total': 0} for name in models_dict}}
    pbar = tqdm(loader, desc='  Running AFA-v3 Attack', unit='batch')

    for x_clean, y_true in pbar:
        x_clean = x_clean.to(wb_device)
        y_true  = y_true.to(wb_device)

        if 'svhn' in str(loader.dataset):
            y_true = y_true % 10

        correct_mask = (get_predictions(wb_model, x_clean).to(wb_device) == y_true)
        if correct_mask.sum() == 0:
            continue
        x_clean, y_true = x_clean[correct_mask], y_true[correct_mask]

        x_adv_afa = afa_attack_v2(
            model=wb_model, x_orig=x_clean, y=y_true,
            gradcam=gradcam,
            epsilon=cfg['epsilon'], T=cfg['T'], N=cfg['N'],
            alpha=cfg['alpha'], rho=cfg['rho'], sigma=cfg['sigma'],
            tau=cfg['tau'], tau_min=cfg['tau_min'],
            tau_decay=cfg.get('tau_decay', 0.5),
            lambda_val=cfg['lambda_coup'], delta_conv=cfg['delta_conv'],
            mu=cfg.get('mu', 1.0),
            use_di=True,
            di_prob=cfg.get('di_prob', 0.7),
            di_scale_low=cfg.get('di_scale_low', 0.90),
            ti_kernel=ti_k,
            use_ni=cfg.get('use_ni', True),
            ni_mu=cfg.get('ni_mu', 1.0),
            surrogates=attack_surrogates,
            surrogate_sample_k=cfg.get('surrogate_sample_k', 1),
            offload_surrogates=cfg.get('offload_surrogates', True),
            white_box_name=white_box_name,
        )

        n_batch = y_true.shape[0]
        for name, model in models_dict.items():
            preds = get_predictions(model, x_adv_afa)
            n_success = (preds != y_true.cpu()).sum().item()
            results['afa'][name]['hits'] += n_success
            results['afa'][name]['total'] += n_batch

    final = {}
    for name in models_dict:
        d = results['afa'][name]
        final[name] = (d['hits'] / d['total'] * 100.0) if d['total'] > 0 else 0.0
    results['afa'] = final

    return results


In [ ]:
# ─────────────────────────────────────────
# SECTION 7b — Run AFA-Attack on All Datasets
# ─────────────────────────────────────────

import time
import numpy as np

ALL_RESULTS = {}
DATASETS = ['cifar10', 'cifar100', 'svhn']

for ds in DATASETS:
    if ds not in LOADERS:
        print(f"❌ Loader missing for {ds}"); continue
    if ds not in ALL_MODELS:
        print(f"❌ Models missing for {ds}"); continue

    print(f"\n{'='*55}")
    print(f"  DATASET: {ds.upper()}")
    print(f"  White-box: {WHITE_BOX}  |  ε={CFG['epsilon']*255:.0f}/255")
    print(f"  T={CFG['T']}  N={CFG['N']}  Images={CFG['num_images']}")
    print(f"{'='*55}")

    active_models = {k: v for k, v in ALL_MODELS[ds].items() if v is not None}

    if WHITE_BOX not in active_models:
        print(f"⚠️ White-box '{WHITE_BOX}' not found → skipping {ds}"); continue
    if len(active_models) < 2:
        print(f"⚠️ Not enough models → skipping {ds}"); continue

    gradcam = GRADCAMS.get(ds, None) if 'GRADCAMS' in globals() else None

    try:
        t0 = time.time()
        results = run_experiment(
            loader=LOADERS[ds], models_dict=active_models,
            white_box_name=WHITE_BOX, gradcam=gradcam, cfg=CFG,
        )
        ALL_RESULTS[ds] = results
        elapsed = time.time() - t0
    except Exception as e:
        print(f"❌ Error on {ds}: {e}"); continue

    res = ALL_RESULTS[ds]
    black_models = [k for k in active_models if k != WHITE_BOX]

    print(f"\n  {'Model':15s}  {'AFA-Attack ASR':>16s}")
    print(f"  {'-'*34}")
    for name in active_models:
        afa = res['afa'].get(name, 0.0)
        tag = " ◀ WB" if name == WHITE_BOX else "   BB"
        print(f"  {name:15s}  {afa:>14.1f}%  {tag}")
    print(f"  {'-'*34}")

    if black_models:
        bb_afa = np.mean([res['afa'][n] for n in black_models])
        print(f"  {'Black-box AVG':15s}  {bb_afa:>14.1f}%")
        print(f"\n  ⏱ Elapsed: {elapsed:.0f}s")


In [ ]:
# ─────────────────────────────────────────
# SECTION 7c — AFA-Attack Results Visualization
# ─────────────────────────────────────────

if not ALL_RESULTS:
    print("⚠️  No results yet — run the experiment cell first.")
else:
    datasets_done = list(ALL_RESULTS.keys())
    n_ds = len(datasets_done)

    fig, axes = plt.subplots(1, n_ds, figsize=(7 * n_ds, 6))
    if n_ds == 1:
        axes = [axes]

    fig.suptitle('AFA-Attack: Attack Success Rate (%) by Dataset',
                 fontsize=14, fontweight='bold', y=1.02)

    for ax, ds in zip(axes, datasets_done):
        res         = ALL_RESULTS[ds]
        model_names = list(res['afa'].keys())
        x_pos = np.arange(len(model_names))
        afa_v = [res['afa'][n] for n in model_names]

        bars = ax.bar(x_pos, afa_v, 0.6, label='AFA-Attack',
                      color='crimson', alpha=0.85, edgecolor='darkred')

        for bar, val in zip(bars, afa_v):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                    f'{val:.1f}', ha='center', va='bottom', fontsize=8, color='darkred')

        ax.set_xticks(x_pos)
        ax.set_xticklabels(
            [f"{n}\n{'WB' if n==WHITE_BOX else 'BB'}" for n in model_names],
            fontsize=8
        )
        ax.set_ylim(0, 115)
        ax.set_ylabel('Attack Success Rate (%)')
        ax.set_title(f'{ds.upper()}\n(white-box: {WHITE_BOX})', fontsize=11)
        ax.legend(fontsize=9)
        ax.grid(axis='y', alpha=0.3)

        wb_idx = model_names.index(WHITE_BOX)
        ax.axvspan(wb_idx - 0.5, wb_idx + 0.5, alpha=0.06, color='gold')

    plt.tight_layout()
    save_path = '/content/Paper_Result.png'
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f"\n✅ Visualization saved to {save_path}")


In [ ]:
# ─────────────────────────────────────────
# SECTION 7d — AFA-Attack ASR Heatmap
# ─────────────────────────────────────────

if ALL_RESULTS:
    datasets_done = list(ALL_RESULTS.keys())
    all_archs     = ARCH_NAMES

    afa_matrix = np.zeros((len(all_archs), len(datasets_done)))

    for j, ds in enumerate(datasets_done):
        res = ALL_RESULTS[ds]
        for i, arch in enumerate(all_archs):
            if arch in res['afa']:
                afa_matrix[i, j] = res['afa'][arch]

    fig, ax = plt.subplots(figsize=(6, 5))
    fig.suptitle('AFA-Attack ASR (%) — All Datasets × Architectures',
                 fontsize=13, fontweight='bold')

    im = ax.imshow(afa_matrix, cmap='Reds', aspect='auto', vmin=0, vmax=100)
    ax.set_xticks(range(len(datasets_done)))
    ax.set_xticklabels([d.upper() for d in datasets_done])
    ax.set_yticks(range(len(all_archs)))
    ax.set_yticklabels(all_archs)
    ax.set_title('AFA-Attack ASR (%)')
    plt.colorbar(im, ax=ax)

    for ii in range(len(all_archs)):
        for jj in range(len(datasets_done)):
            ax.text(jj, ii, f'{afa_matrix[ii,jj]:.1f}',
                    ha='center', va='center', fontsize=9,
                    color='white' if afa_matrix[ii,jj] > 60 else 'black')

    plt.tight_layout()
    save_path = '/content/Paper_afa_heatmap.png'
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    print(f"✅ Heatmap saved to {save_path}")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

def visualize_attack_comparison(
    clean_imgs,
    adv_imgs,
    true_labels,
    clean_preds,
    adv_preds,
    class_names,
    dataset_name="Dataset",
    num_images=5,
    save_path="/kaggle/working/PAPER_RUN",        # 👉 NEW
    dpi=1000               # 👉 NEW
):
    """
    Visualize clean vs adversarial images + optionally save at high DPI
    """

    # Limit images
    clean_imgs = clean_imgs[:num_images]
    adv_imgs   = adv_imgs[:num_images]

    # Convert to numpy
    clean_imgs = clean_imgs.permute(0,2,3,1).numpy()
    adv_imgs   = adv_imgs.permute(0,2,3,1).numpy()

    fig, axes = plt.subplots(3, num_images, figsize=(3*num_images, 8))

    for i in range(num_images):

        t = int(true_labels[i])
        cp = int(clean_preds[i])
        ap = int(adv_preds[i])

        # --- CLEAN ---
        axes[0, i].imshow(np.clip(clean_imgs[i], 0, 1))
        axes[0, i].set_title(
            f"Clean\nT:{class_names[t]}\nP:{class_names[cp]}",
            fontsize=10
        )
        axes[0, i].axis('off')

        # --- ADV ---
        success = (cp != ap)
        axes[1, i].imshow(np.clip(adv_imgs[i], 0, 1))
        axes[1, i].set_title(
            f"Adv\nP:{class_names[ap]}",
            fontsize=10,
            color='red' if success else 'green'
        )
        axes[1, i].axis('off')

        # --- PERTURBATION ---
        perturb = np.abs(adv_imgs[i] - clean_imgs[i])
        perturb = perturb / (perturb.max() + 1e-8)

        axes[2, i].imshow(perturb)
        axes[2, i].set_title("Perturbation", fontsize=10)
        axes[2, i].axis('off')

    # Labels
    axes[0, 0].set_ylabel("Clean", fontsize=12)
    axes[1, 0].set_ylabel("Adversarial", fontsize=12)
    axes[2, 0].set_ylabel("Perturbation", fontsize=12)

    plt.suptitle(f"{dataset_name}: Clean vs AFA Attack", fontsize=14)
    plt.tight_layout()

    # ✅ SAVE (1000 DPI)
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"✅ Saved high-quality image at: {save_path} (DPI={dpi})")

    plt.show()

In [25]:
# ─────────────────────────────────────────
# 🚀 MAIN EXECUTION
# ─────────────────────────────────────────
ds = 'cifar10'   # 👉 change to 'cifar100' or 'svhn'

model = ALL_MODELS[ds][WHITE_BOX]
model.eval()
dev   = next(model.parameters()).device

# Load a batch — keep at 32×32, NormalizedModel handles the rest
images, labels = next(iter(LOADERS[ds]))
images, labels = images.to(dev), labels.to(dev)

if ds == 'svhn':
    labels = labels % 10

class_names = get_class_names(LOADERS[ds], ds)

# Clean predictions (feed 32×32 directly)
with torch.no_grad():
    _, clean_preds = model(images).max(1)

gradcam = GRADCAMS.get(ds) if 'GRADCAMS' in globals() else None
active_models = {k: v for k, v in ALL_MODELS[ds].items() if v is not None}
ti_k = TI_KERNEL.to(dev) if 'TI_KERNEL' in globals() and TI_KERNEL is not None else None

adv_images = afa_attack_v2(
    model=model, x_orig=images, y=labels,
    gradcam=gradcam,
    epsilon=CFG['epsilon'], T=CFG['T'], N=CFG['N'],
    alpha=CFG['alpha'], rho=CFG['rho'], sigma=CFG['sigma'],
    tau=CFG['tau'], tau_min=CFG['tau_min'], tau_decay=CFG.get('tau_decay', 0.5),
    lambda_val=CFG['lambda_coup'], delta_conv=CFG['delta_conv'],
    mu=CFG.get('mu', 1.0),
    use_di=True, di_prob=CFG.get('di_prob', 0.7),
    di_scale_low=CFG.get('di_scale_low', 0.90),
    ti_kernel=ti_k,
    use_ni=CFG.get('use_ni', True),
    ni_mu=CFG.get('ni_mu', 1.0),
    surrogates=active_models,
    surrogate_sample_k=CFG.get('surrogate_sample_k', 1),
    offload_surrogates=CFG.get('offload_surrogates', True),
    white_box_name=WHITE_BOX,
).detach()

with torch.no_grad():
    _, adv_preds = model(adv_images).max(1)

if ds == 'svhn':
    clean_preds, adv_preds = clean_preds % 10, adv_preds % 10

visualize_attack_comparison(
    clean_imgs=images.cpu(), adv_imgs=adv_images.cpu(),
    true_labels=labels.cpu(), clean_preds=clean_preds.cpu(),
    adv_preds=adv_preds.cpu(), class_names=class_names,
    dataset_name=ds.upper(), num_images=5, dpi=1000
)


NameError: name 'get_class_names' is not defined

In [ ]:
# ─────────────────────────────────────────
# SECTION 9a — Mount Google Drive
# ─────────────────────────────────────────

from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/AFA_Attack_Results'
import os
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"✅ Drive mounted. Results will be saved to:\n   {DRIVE_SAVE_DIR}")


In [ ]:
# ─────────────────────────────────────────
# SECTION 9b — Copy Output Files to Drive
# ─────────────────────────────────────────

import shutil, os, glob

output_files = (glob.glob('/content/Paper_*.png') +
                glob.glob('/content/Paper_*.csv') +
                glob.glob('/content/Paper_*.xlsx'))

for fpath in output_files:
    dest = os.path.join(DRIVE_SAVE_DIR, os.path.basename(fpath))
    shutil.copy2(fpath, dest)
    print(f"  ✅ Copied: {os.path.basename(fpath)}")

if not output_files:
    print("⚠️  No output files found — run earlier cells first.")
else:
    print(f"\n✅ {len(output_files)} file(s) saved to Google Drive.")


In [ ]:
# ─────────────────────────────────────────
# SECTION 9c — Zip & Save Full Output Archive to Drive
# ─────────────────────────────────────────

import shutil, os

zip_base = '/content/AFA_Attack_full_output'
shutil.make_archive(base_name=zip_base, format='zip',
                    root_dir='/content', base_dir='.')

dest_zip = os.path.join(DRIVE_SAVE_DIR, 'AFA_Attack_full_output.zip')
shutil.copy2(zip_base + '.zip', dest_zip)
print(f"✅ Archive saved to Drive:\n   {dest_zip}")


In [ ]:
# ─────────────────────────────────────────
# SECTION 9d — Verify Drive Contents
# ─────────────────────────────────────────

import os

print(f"📁 Files in {DRIVE_SAVE_DIR}:\n")
for fname in sorted(os.listdir(DRIVE_SAVE_DIR)):
    fpath = os.path.join(DRIVE_SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1024
    print(f"  {fname:45s}  {size:8.1f} KB")

print("\n✅ All done!")
